# Inference Ablation Study: Training Features vs Pipeline Features

Isolate which conditioning signal(s) cause the quality gap (especially consonant clarity)
between the **notebook path** (pre-computed `.npy` features from training data) and the
**pipeline path** (features re-derived from USTX + prior WAV).

**Ablation conditions** (starting from known-good training features, replacing one at a time):
1. **All Training** — baseline, all pre-computed `.npy` files
2. **Pipeline Phonemes** — replace phoneme_ids with pipeline-derived from USTX
3. **Pipeline PL-BERT** — replace PL-BERT features with pipeline-derived from USTX
4. **Pipeline F0+Voicing (USTX)** — replace F0 with USTX pitch curves, voicing with note coverage
5. **All Pipeline** — all signals pipeline-derived (USTX default)
6. **Single-pass (256)** — all training features, cropped/padded to exactly 256 frames,
   single `sample_ode` call (no chunking). Matches training input format exactly.

**Model**: 4-25-big / checkpoint_105000

In [ ]:
# ── Configuration ──

CHECKPOINT_PATH = "../VocaloFlow/checkpoints/4-25-big/checkpoint_175000.pt"
MANIFEST_PATH = "../Data/Rachie/manifest.csv"
DATA_DIR = "../Data/Rachie"
PHONESET_PATH = "../SoulX-Singer/soulxsinger/utils/phoneme/phone_set.json"
PLBERT_DIR = "../PL-BERT"

DEVICE = "auto"
N_SAMPLES = 4

# Validation split (must match training config)
MAX_DTW_COST = 100.0
VAL_FRACTION = 0.05
SEED = 42

# ODE inference — identical across all conditions for fair comparison.
# CFG_SCALE=1.0 avoids the confound of the unconditional path also changing
# when we swap signals.
NUM_ODE_STEPS = 16
ODE_METHOD = "midpoint"
CFG_SCALE = 1.0
CHUNK_SIZE = 256
OVERLAP = 16
TIME_SCHEDULE = "sway"
SWAY_COEFF = -1.0

MAX_SEQ_LEN = 256  # training max_seq_len for condition 6

SR = 24000
HOP = 480

# DTW cost range for sample selection
SAMPLE_DTW_MIN = 20.0
SAMPLE_DTW_MAX = 60.0

SAMPLE_SEED = 1

In [ ]:
import sys, os, math
from pathlib import Path
from collections import OrderedDict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML

# ── Path setup (mirrors VocaloFlow/inference/pipeline.py) ──
REPO_ROOT = str(Path("..").resolve())
VOCALOFLOW_DIR = os.path.join(REPO_ROOT, "VocaloFlow")
SOULX_DIR = os.path.join(REPO_ROOT, "SoulX-Singer")

for p in [REPO_ROOT, VOCALOFLOW_DIR, SOULX_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from utils.data_helpers import load_manifest, filter_manifest, split_by_song
from utils.resample import resolve_phoneme_indirection, resample_1d, resample_2d

from inference.pipeline import (
    load_model, infer_chunked, mel_to_wav,
    parse_ustx, synthesize_f0_from_ustx_pitch,
    get_note_coverage_voicing,
    build_phoneme_ids, build_plbert_frame_features,
)
from inference.inference import sample_ode

if DEVICE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(DEVICE)
print(f"Using device: {device}")

In [ ]:
# ── Load manifest and select validation samples ──

manifest = load_manifest(MANIFEST_PATH, DATA_DIR)
filtered = filter_manifest(manifest, max_dtw_cost=MAX_DTW_COST)
train_df, val_df = split_by_song(filtered, val_fraction=VAL_FRACTION, seed=SEED)

candidates = val_df[
    (val_df["dtw_cost"] >= SAMPLE_DTW_MIN) &
    (val_df["dtw_cost"] <= SAMPLE_DTW_MAX)
].copy()

def _has_pipeline_inputs(row):
    d = os.path.dirname(row["prior_mel_path"])
    return all(os.path.exists(os.path.join(d, f))
               for f in ("prior.ustx", "prior.wav", "plbert_features.npy"))

candidates = candidates[candidates.apply(_has_pipeline_inputs, axis=1)]
samples = candidates.sample(n=min(N_SAMPLES, len(candidates)), random_state=SAMPLE_SEED)

print(f"Validation set: {len(val_df)} chunks")
print(f"Candidates (DTW {SAMPLE_DTW_MIN}-{SAMPLE_DTW_MAX}, pipeline inputs present): {len(candidates)}")
print(f"\nSelected {len(samples)} samples:")
for _, row in samples.iterrows():
    print(f"  {row['dali_id'][:12]}…/{row['chunk_name']}  DTW={row['dtw_cost']:.1f}")

In [ ]:
# ── Load model ──

model = load_model(CHECKPOINT_PATH, device)
print(f"use_plbert: {model.config.use_plbert}")
print(f"use_speaker_embedding: {model.config.use_speaker_embedding}")

speaker_embedding = None

In [ ]:
# ── Helper functions ──

def _match_1d(x, n):
    """Truncate or zero-pad a 1-D array to length *n*."""
    return x[:n] if len(x) >= n else np.pad(x, (0, n - len(x)))

def _match_2d(x, n):
    """Truncate or zero-pad a 2-D array along axis 0 to length *n*."""
    return x[:n] if x.shape[0] >= n else np.pad(x, ((0, n - x.shape[0]), (0, 0)))


def load_training_inputs(row):
    """Load all pre-computed training features for a validation chunk.

    Returns dict with prior_mel, target_mel, f0, voicing, phoneme_ids,
    plbert_features, chunk_dir, T — all matched to target_mel's length.
    """
    prior_mel = np.load(row["prior_mel_path"]).astype(np.float32)
    target_mel = np.load(row["target_mel_path"]).astype(np.float32)
    f0 = np.load(row["f0_path"]).astype(np.float32)
    voicing = np.load(row["voicing_path"]).astype(np.float32)

    chunk_dir = os.path.dirname(row["prior_mel_path"])
    phoneme_mask = np.load(row["phoneme_mask_path"]).astype(np.int64)
    phoneme_ids_raw = np.load(os.path.join(chunk_dir, "phoneme_ids.npy")).astype(np.int64)
    resolved = resolve_phoneme_indirection(phoneme_ids_raw, phoneme_mask)

    T = target_mel.shape[0]
    prior_mel = _match_2d(prior_mel, T)
    f0 = _match_1d(f0, T)
    voicing = _match_1d(voicing, T)
    resolved = _match_1d(resolved, T)

    plbert_feats = np.load(os.path.join(chunk_dir, "plbert_features.npy")).astype(np.float32)
    mask_clipped = np.clip(phoneme_mask, 0, len(plbert_feats) - 1)
    plbert_frame = _match_2d(plbert_feats[mask_clipped], T)

    return {
        "prior_mel": prior_mel, "target_mel": target_mel,
        "f0": f0, "voicing": voicing, "phoneme_ids": resolved,
        "plbert_features": plbert_frame, "chunk_dir": chunk_dir, "T": T,
    }


def derive_pipeline_inputs(chunk_dir, T):
    """Re-derive all conditioning signals from USTX (new default).

    Uses USTX pitch curves for F0 and note-coverage for voicing,
    matching the pipeline's default --f0-mode ustx.
    All outputs resampled to length *T* for fair comparison.
    """
    ustx_path = os.path.join(chunk_dir, "prior.ustx")
    phoneset = os.path.abspath(PHONESET_PATH)

    notes_data = parse_ustx(ustx_path)
    notes, ms_per_tick = notes_data["notes"], notes_data["ms_per_tick"]

    pipe_phonemes = build_phoneme_ids(notes, ms_per_tick, T, phoneset)
    pipe_plbert = build_plbert_frame_features(
        notes, ms_per_tick, T, phoneset,
        plbert_dir=os.path.abspath(PLBERT_DIR), device=str(device),
    )
    pipe_voicing = get_note_coverage_voicing(notes_data, T)
    pipe_f0 = synthesize_f0_from_ustx_pitch(notes_data, T, voicing=pipe_voicing)
    pipe_f0 = resample_1d(pipe_f0, T, mode="linear").numpy().astype(np.float32)

    return {
        "phoneme_ids": pipe_phonemes, "plbert_features": pipe_plbert,
        "f0": pipe_f0, "voicing": pipe_voicing,
    }

print("Helpers defined.")

In [ ]:
# ── Ablation condition builder ──

CONDITION_NAMES = [
    "1. All Training",
    "2. Pipeline Phonemes",
    "3. Pipeline PL-BERT",
    "4. Pipeline F0+Voicing",
    "5. All Pipeline",
    "6. Single-pass (256)",
]


def build_ablation_conditions(training, pipeline):
    """Build the 6 ablation condition dicts.

    Conditions 1-5 return signal dicts for use with infer_chunked.
    Condition 6 is handled specially in the compute loop (single-pass).
    """
    base = {
        "prior_mel": training["prior_mel"],
        "f0": training["f0"],
        "voicing": training["voicing"],
        "phoneme_ids": training["phoneme_ids"],
        "plbert_features": training["plbert_features"],
    }

    conditions = OrderedDict()
    conditions["1. All Training"] = dict(base)

    c2 = dict(base)
    c2["phoneme_ids"] = pipeline["phoneme_ids"]
    conditions["2. Pipeline Phonemes"] = c2

    c3 = dict(base)
    c3["plbert_features"] = pipeline["plbert_features"]
    conditions["3. Pipeline PL-BERT"] = c3

    c4 = dict(base)
    c4["f0"] = pipeline["f0"]
    c4["voicing"] = pipeline["voicing"]
    conditions["4. Pipeline F0+Voicing"] = c4

    conditions["5. All Pipeline"] = {
        "prior_mel": training["prior_mel"],
        "f0": pipeline["f0"],
        "voicing": pipeline["voicing"],
        "phoneme_ids": pipeline["phoneme_ids"],
        "plbert_features": pipeline["plbert_features"],
    }

    # Condition 6 placeholder — actual forward pass is handled in compute loop
    conditions["6. Single-pass (256)"] = dict(base)

    return conditions


def run_single_pass_256(model, cond, device):
    """Condition 6: crop/pad to 256, single sample_ode call, no chunking.

    Replicates exactly what the model sees during training.
    """
    L = MAX_SEQ_LEN
    pm = _match_2d(cond["prior_mel"], L)
    f = _match_1d(cond["f0"], L)
    v = _match_1d(cond["voicing"], L)
    ph = _match_1d(cond["phoneme_ids"], L)
    pl = _match_2d(cond["plbert_features"], L)

    length = min(cond["prior_mel"].shape[0], L)
    mask = torch.zeros(1, L, dtype=torch.bool, device=device)
    mask[0, :length] = True

    pm_t = torch.from_numpy(pm.astype(np.float32)).unsqueeze(0).to(device)
    f_t = torch.from_numpy(f.astype(np.float32)).unsqueeze(0).to(device)
    v_t = torch.from_numpy(v.astype(np.float32)).unsqueeze(0).to(device)
    ph_t = torch.from_numpy(ph.astype(np.int64)).unsqueeze(0).to(device)
    pl_t = torch.from_numpy(pl.astype(np.float32)).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = sample_ode(
            model, pm_t, f_t, v_t, ph_t,
            num_steps=NUM_ODE_STEPS, method=ODE_METHOD, padding_mask=mask,
            cfg_scale=CFG_SCALE, plbert_features=pl_t,
            speaker_embedding=speaker_embedding,
            time_schedule=TIME_SCHEDULE, sway_coeff=SWAY_COEFF,
        )
    return pred[0, :length].cpu().numpy()

print("Condition builder defined.")

In [ ]:
# ── Run ablation inference ──

all_results = []

for idx, (_, row) in enumerate(samples.iterrows()):
    sample_id = f"{row['dali_id'][:12]}…/{row['chunk_name']}"
    print(f"\n{'=' * 70}")
    print(f"Sample {idx + 1}/{len(samples)}: {sample_id}  (DTW={row['dtw_cost']:.1f})")
    print(f"{'=' * 70}")

    training = load_training_inputs(row)
    T = training["T"]
    print(f"Target length: {T} frames ({T * HOP / SR:.2f}s)")

    print("\n── Deriving pipeline features ──")
    pipeline = derive_pipeline_inputs(training["chunk_dir"], T)

    # Signal divergence summary
    ph_agree = np.mean(training["phoneme_ids"] == pipeline["phoneme_ids"])
    f0_corr = np.corrcoef(training["f0"], pipeline["f0"])[0, 1] if T > 1 else 0
    voicing_agree = np.mean(training["voicing"] == pipeline["voicing"])
    cos_sims = np.array([
        np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
        for a, b in zip(training["plbert_features"], pipeline["plbert_features"])
    ])
    print(f"\n── Signal divergences ──")
    print(f"  Phoneme ID agreement:        {ph_agree:.1%}")
    print(f"  F0 correlation:              {f0_corr:.4f}")
    print(f"  Voicing agreement:           {voicing_agree:.1%}")
    print(f"  PL-BERT mean cosine sim:     {cos_sims.mean():.4f}")

    conditions = build_ablation_conditions(training, pipeline)

    result = {
        "sample_id": sample_id,
        "row": row,
        "training": training,
        "pipeline": pipeline,
        "target_mel": training["target_mel"],
        "output_mels": OrderedDict(),
        "output_audios": OrderedDict(),
    }

    for cond_name, cond in conditions.items():
        print(f"\n── {cond_name} ──")

        if cond_name == "6. Single-pass (256)":
            out_mel = run_single_pass_256(model, cond, device)
        else:
            with torch.no_grad():
                out_mel = infer_chunked(
                    model, cond["prior_mel"], cond["f0"], cond["voicing"],
                    cond["phoneme_ids"],
                    chunk_size=CHUNK_SIZE, overlap=OVERLAP,
                    num_steps=NUM_ODE_STEPS, method=ODE_METHOD,
                    device=device, cfg_scale=CFG_SCALE,
                    plbert_features=cond["plbert_features"],
                    speaker_embedding=speaker_embedding,
                    time_schedule=TIME_SCHEDULE, sway_coeff=SWAY_COEFF,
                )

        result["output_mels"][cond_name] = out_mel
        result["output_audios"][cond_name] = mel_to_wav(out_mel)

    all_results.append(result)

print(f"\n{'=' * 70}")
print(f"All {len(all_results)} samples processed.")

In [ ]:
# ── Mel spectrogram comparison ──

n_conds = len(CONDITION_NAMES)

for result in all_results:
    target_mel = result["target_mel"]
    prior_mel = result["training"]["prior_mel"]
    T = target_mel.shape[0]
    t_end = T * HOP / SR

    display(HTML(f"<h2>{result['sample_id']}</h2>"))

    # Row 1: mel spectrograms (Target, Prior, + 6 conditions)
    all_mels = [target_mel, prior_mel]
    all_titles = ["Target (Ground Truth)", "Prior (OpenUtau)"]
    for cn in CONDITION_NAMES:
        mel = result["output_mels"][cn]
        all_mels.append(_match_2d(mel, T))
        all_titles.append(cn)

    vmin = min(m.min() for m in all_mels)
    vmax = max(m.max() for m in all_mels)
    n_panels = len(all_mels)
    fig, axes = plt.subplots(1, n_panels, figsize=(3.5 * n_panels, 4))
    fig.suptitle(f"Mel Spectrograms — {result['sample_id']}", fontsize=12)
    for ax, mel, title in zip(axes, all_mels, all_titles):
        te = mel.shape[0] * HOP / SR
        im = ax.imshow(mel.T, origin="lower", aspect="auto", cmap="magma",
                        vmin=vmin, vmax=vmax, extent=[0, te, 0, 128])
        ax.set_title(title, fontsize=8)
        ax.set_xlabel("Time (s)", fontsize=7)
        ax.set_ylabel("Mel bin", fontsize=7)
        ax.tick_params(labelsize=6)
    fig.colorbar(im, ax=list(axes), shrink=0.7)
    plt.tight_layout()
    plt.show()

    # Row 2: difference from target
    fig, axes = plt.subplots(1, n_conds, figsize=(3.5 * n_conds, 4))
    fig.suptitle(f"Difference from Target — {result['sample_id']}", fontsize=12)
    for ax, cn in zip(axes, CONDITION_NAMES):
        mel_out = result["output_mels"][cn]
        L = min(mel_out.shape[0], T)
        diff = _match_2d(mel_out, L) - target_mel[:L]
        vlim = max(np.abs(diff).max(), 0.01)
        te = L * HOP / SR
        im = ax.imshow(diff.T, origin="lower", aspect="auto", cmap="coolwarm",
                        vmin=-vlim, vmax=vlim, extent=[0, te, 0, 128])
        mae = np.abs(diff).mean()
        ax.set_title(f"{cn}\nMAE={mae:.4f}", fontsize=8)
        ax.set_xlabel("Time (s)", fontsize=7)
        ax.set_ylabel("Mel bin", fontsize=7)
        ax.tick_params(labelsize=6)
        fig.colorbar(im, ax=ax, shrink=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Quantitative metrics table + bar chart ──

summary = {c: {"mae": [], "mse": []} for c in CONDITION_NAMES}

print(f"{'Condition':<25} | {'Sample':<30} | {'MAE':>8} | {'MSE':>8}")
print("-" * 80)

for result in all_results:
    target = result["target_mel"]
    T = target.shape[0]
    for cn in CONDITION_NAMES:
        mel_out = result["output_mels"][cn]
        L = min(mel_out.shape[0], T)
        diff = _match_2d(mel_out, L) - target[:L]
        mae = np.abs(diff).mean()
        mse = (diff ** 2).mean()
        summary[cn]["mae"].append(mae)
        summary[cn]["mse"].append(mse)
        print(f"{cn:<25} | {result['sample_id']:<30} | {mae:8.4f} | {mse:8.4f}")

print("=" * 80)
baseline_mae = np.mean(summary["1. All Training"]["mae"])

print(f"\n{'Condition':<25} | {'Mean MAE':>10} | {'Mean MSE':>10} | {'Delta MAE':>12}")
print("-" * 70)
for cn in CONDITION_NAMES:
    m_mae = np.mean(summary[cn]["mae"])
    m_mse = np.mean(summary[cn]["mse"])
    delta = m_mae - baseline_mae
    sign = "+" if delta > 0 else ""
    print(f"{cn:<25} | {m_mae:10.4f} | {m_mse:10.4f} | {sign}{delta:10.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
means = [np.mean(summary[c]["mae"]) for c in CONDITION_NAMES]
colors = ["#2ecc71", "#e74c3c", "#e67e22", "#3498db", "#9b59b6", "#1abc9c"]
bars = ax.bar(range(n_conds), means, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(range(n_conds))
ax.set_xticklabels(CONDITION_NAMES, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean MAE vs Target")
ax.set_title("Ablation: Mean MAE by Condition (lower = closer to target)")
ax.axhline(means[0], color="gray", linestyle="--", alpha=0.5, label="Baseline")
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=7)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Audio playback ──

for result in all_results:
    display(HTML(f"<h3>{result['sample_id']}</h3>"))
    chunk_dir = result["training"]["chunk_dir"]

    print("Prior (OpenUtau):")
    display(Audio(os.path.join(chunk_dir, "prior.wav")))
    print("Target (Ground Truth):")
    display(Audio(os.path.join(chunk_dir, "target.wav")))

    for cn in CONDITION_NAMES:
        print(f"{cn}:")
        display(Audio(result["output_audios"][cn], rate=SR))

    display(HTML("<hr>"))

In [ ]:
# ── Signal divergence diagnostic plots ──

for result in all_results:
    training = result["training"]
    pipeline = result["pipeline"]
    T = training["T"]
    t_axis = np.arange(T) * HOP / SR

    display(HTML(f"<h3>Signal Divergences — {result['sample_id']}</h3>"))
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    # 1. Phoneme IDs
    ax = axes[0]
    ax.plot(t_axis, training["phoneme_ids"], label="Training", alpha=0.7)
    ax.plot(t_axis, pipeline["phoneme_ids"], label="Pipeline", alpha=0.7, linestyle="--")
    mismatch = training["phoneme_ids"] != pipeline["phoneme_ids"]
    if mismatch.any():
        ylo, yhi = ax.get_ylim()
        ax.fill_between(t_axis, ylo, yhi, where=mismatch,
                         alpha=0.12, color="red", label="Mismatch")
        ax.set_ylim(ylo, yhi)
    ax.set_ylabel("Phoneme ID")
    ax.set_title(f"Phoneme IDs — agreement: {(~mismatch).mean():.1%}")
    ax.legend(fontsize=8)

    # 2. F0 contours
    ax = axes[1]
    ax.plot(t_axis, training["f0"], label="Training (target RMVPE)", alpha=0.8)
    ax.plot(t_axis, pipeline["f0"], label="Pipeline (USTX pitch curves)", alpha=0.8, linestyle="--")
    corr = np.corrcoef(training["f0"], pipeline["f0"])[0, 1]
    ax.set_ylabel("F0 (Hz)")
    ax.set_title(f"F0 Contours — correlation: {corr:.4f}")
    ax.legend(fontsize=8)

    # 3. PL-BERT cosine similarity per frame
    ax = axes[2]
    cos_per_frame = np.array([
        np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
        for a, b in zip(training["plbert_features"], pipeline["plbert_features"])
    ])
    ax.plot(t_axis, cos_per_frame, color="purple", alpha=0.8)
    ax.axhline(1.0, color="gray", linestyle="--", alpha=0.3)
    ax.set_ylim(-0.1, 1.1)
    ax.set_ylabel("Cosine Similarity")
    ax.set_xlabel("Time (s)")
    ax.set_title(f"PL-BERT Frame Cosine Similarity — mean: {cos_per_frame.mean():.4f}")

    plt.tight_layout()
    plt.show()

## Findings

| Condition | Delta MAE | Interpretation |
|-----------|-----------|----------------|
| 1. All Training (baseline) | 0.0000 | Reference quality |
| 2. Pipeline Phonemes | +???? | Impact of re-deriving phoneme IDs from USTX |
| 3. Pipeline PL-BERT | +???? | Impact of live PL-BERT extraction vs pre-computed |
| 4. Pipeline F0+Voicing (USTX) | +???? | Impact of USTX pitch curves + note-coverage voicing vs target RMVPE |
| 5. All Pipeline (USTX default) | +???? | Combined impact of all pipeline signal derivation |
| 6. Single-pass (256) | +???? | Impact of chunking vs single forward pass |

**Biggest contributor to quality gap**: (fill after running)

**Consonant clarity**: (fill — listen to conditions 2 and 3 carefully)

**Next steps**: (e.g., fix the dominant signal, or investigate why it diverges)